In [ ]:
# IMPORT DATA AND MAP SENTIMENT
# Note: we trained the model on 2 GPUs
import pandas as pd
import os

df = pd.read_csv("/kaggle/input/datasets/mikavanstraaten/dsp-data/reviews.csv")  # only contains the label, text, type and an id column
df_split = pd.read_csv("/kaggle/input/datasets/mikavanstraaten/data-split/data_split.csv")  # train-validation-test split
df = pd.concat([df, df_split], axis=1)

def map_label(row):
    if row["type"].lower() == "review":
        if row["label"] in [1, 2]:
            return 0
        elif row["label"] == 3:
            return 1
        elif row["label"] in [4, 5]:
            return 2
        else:
            return None
    else:
        if row["label"] == -1:
            return 0
        elif row["label"] == 0:
            return 1
        elif row["label"] == 1:
            return 2
        else:
            return Noneiligheid

df["sentiment_encoded"] = df.apply(map_label, axis=1)
df["sentiment_encoded"] = df["sentiment_encoded"].astype(int)
df = df[(df["type"] != "Tweet") | (df["text"].str.len() <= 200)]  # delete 6 outlier tweets
df = df.drop(columns=["Unnamed: 1", "Unnamed: 0", "label"])

df.head()

In [ ]:
# PREPROCESS TEXT
from bs4 import BeautifulSoup
import re
import emoji

def clean_text(x):
    x = BeautifulSoup(x, "html.parser").get_text()  # delete all html tags
    x = re.sub(r"http\S+|www\.\S+", "URL", x)  # change URLs to a dedicated URL token
    x = re.sub(r"@\w+", "MNTN", x)  # change @mentions to a dedicated MNTN token
    x = emoji.demojize(x, delimiters=(" ", " ")).replace("_", " ")  # replace emojis with text
    x = re.sub(r'\s+', ' ', x)  # delete multiple spaces
    return x.lower()

df["review_cleaned"] = df["text"].apply(clean_text)
df = df.drop(columns=["text"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

In [ ]:
# MODEL CLASS
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup)


class FineTuningPipeline:

    def __init__(
            self,
            dataset,
            tokenizer,
            model,
            val_size = 0.15,
            epochs = 4,
            seed = 42):

        self.df_dataset = dataset
        self.tokenizer = tokenizer
        self.model = model
        self.val_size = val_size
        self.epochs = epochs
        self.seed = seed

        # Check if GPU is available for faster training time
        if torch.cuda.is_available():
            self.device = torch.device('cuda:0')
            self.model.to(self.device)
            if torch.cuda.device_count() > 1:
                self.model = nn.DataParallel(self.model)
                print(f"Using {torch.cuda.device_count()} GPUs")
            else:
                print("Using GPU")
        else:
            self.device = torch.device('cpu')
            self.model.to(self.device)
            print("Using CPU")

        # To handle class imbalance
        class_counts = self.df_dataset['sentiment_encoded'].value_counts().sort_index().values
        weights = 1.0 / class_counts
        weights = weights / weights.sum()
        tensor_weights = torch.tensor(weights, dtype=torch.float).to(self.device)
        self.loss_function = nn.CrossEntropyLoss(weight=tensor_weights)
        
        # Perform fine-tuning        
        self.optimizer = AdamW(self.model.parameters(), lr=2e-5)
        self.set_seeds()
        self.token_ids, self.attention_masks = self.tokenize_dataset()
        self.train_dataloader, self.val_dataloader = self.create_dataloaders()
        self.scheduler = self.create_scheduler()
        self.fine_tune()

    def tokenize(self, text):
        batch_encoder = self.tokenizer(
            text,
            max_length = 512,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt')

        token_ids = batch_encoder['input_ids']
        attention_mask = batch_encoder['attention_mask']

        return token_ids, attention_mask

    def tokenize_dataset(self):
        token_ids = []
        attention_masks = []

        for review in self.df_dataset['review_cleaned']:
            tokens, masks = self.tokenize(review)
            token_ids.append(tokens)
            attention_masks.append(masks)

        token_ids = torch.cat(token_ids, dim=0)
        attention_masks = torch.cat(attention_masks, dim=0)

        return token_ids, attention_masks

    def create_dataloaders(self):     
        train_mask = (self.df_dataset['split'] == 'train').values  # voor split_csv
        val_mask = (self.df_dataset['split'] == 'val').values
        labels = torch.tensor(self.df_dataset['sentiment_encoded'].values, dtype=torch.long)
        train_ids = self.token_ids[train_mask]
        val_ids = self.token_ids[val_mask]
        train_masks = self.attention_masks[train_mask]
        val_masks = self.attention_masks[val_mask]
        train_labels = labels[train_mask]
        val_labels = labels[val_mask]

        train_data = TensorDataset(train_ids, train_masks, train_labels)
        train_dataloader = DataLoader(train_data, shuffle=True, batch_size=32)
        val_data = TensorDataset(val_ids, val_masks, val_labels)
        val_dataloader = DataLoader(val_data, batch_size=32)

        return train_dataloader, val_dataloader

    def create_scheduler(self):
        num_training_steps = self.epochs * len(self.train_dataloader)
        scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=0,
            num_training_steps=num_training_steps)

        return scheduler

    def set_seeds(self):
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        torch.cuda.manual_seed_all(self.seed)

    def fine_tune(self):
        loss_dict = {
            'epoch': [i+1 for i in range(self.epochs)],
            'average training loss': [],
            'average validation loss': []
        }

        t0_train = datetime.now()
        scaler = torch.amp.GradScaler('cuda')

        for epoch in range(0, self.epochs):

            # Train step
            self.model.train()
            training_loss = 0
            t0_epoch = datetime.now()

            print(f'{"-"*20} Epoch {epoch+1} {"-"*20}')
            print('\nTraining:\n---------')
            print(f'Start Time:       {t0_epoch}')

            batch_count = 0
            for batch in self.train_dataloader:

                batch_token_ids = batch[0].to(self.device)
                batch_attention_mask = batch[1].to(self.device)
                batch_labels = batch[2].to(self.device)

                self.model.zero_grad()

                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = self.model(
                        batch_token_ids,
                        token_type_ids = None,
                        attention_mask=batch_attention_mask,
                        return_dict=False)[0]
                    loss = self.loss_function(logits, batch_labels)

                training_loss += loss.item()
                scaler.scale(loss).backward()
                scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                scaler.step(self.optimizer)
                scaler.update()
                self.scheduler.step()
                batch_count+=1
                if batch_count % 100 == 0:
                    print(f"Batch {batch_count} done.")

            average_train_loss = training_loss / len(self.train_dataloader)
            time_epoch = datetime.now() - t0_epoch

            print(f'Average Loss:     {average_train_loss}')
            print(f'Time Taken:       {time_epoch}')

            # Validation step
            self.model.eval()
            val_loss = 0
            val_accuracy = 0
            t0_val = datetime.now()

            print('\nValidation:\n---------')
            print(f'Start Time:       {t0_val}')

            for batch in self.val_dataloader:

                batch_token_ids = batch[0].to(self.device)
                batch_attention_mask = batch[1].to(self.device)
                batch_labels = batch[2].to(self.device)

                with torch.no_grad():
                    logits = self.model(
                        batch_token_ids,
                        attention_mask = batch_attention_mask,
                        token_type_ids = None,
                        return_dict=False)[0]
                    loss = self.loss_function(logits, batch_labels)

                logits = logits.detach().cpu().numpy()
                label_ids = batch_labels.to('cpu').numpy()
                val_loss += loss.item()
                val_accuracy += self.calculate_accuracy(logits, label_ids)


            average_val_accuracy = val_accuracy / len(self.val_dataloader)
            average_val_loss = val_loss / len(self.val_dataloader)
            time_val = datetime.now() - t0_val

            print(f'Average Loss:     {average_val_loss}')
            print(f'Average Accuracy: {average_val_accuracy}')
            print(f'Time Taken:       {time_val}\n')

            loss_dict['average training loss'].append(average_train_loss)
            loss_dict['average validation loss'].append(average_val_loss)

        print(f'Total training time: {datetime.now()-t0_train}')

    def calculate_accuracy(self, preds, labels):
        pred_flat = np.argmax(preds, axis=1).flatten()
        labels_flat = labels.flatten()
        accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

        return accuracy

    def predict(self, dataloader):
        self.model.eval()
        all_logits = []

        for batch in dataloader:

            batch_token_ids, batch_attention_mask = tuple(t.to(self.device) \
                for t in batch)[:2]

            with torch.no_grad():
                logits = self.model(batch_token_ids, attention_mask=batch_attention_mask, return_dict=False)[0]

            all_logits.append(logits)

        all_logits = torch.cat(all_logits, dim=0)

        probs = F.softmax(all_logits, dim=1).cpu().numpy()
        return probs

In [ ]:
# START TRAINING
dataset = df

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Fine-tune model using class
fine_tuned_model = FineTuningPipeline(
    dataset = dataset,
    tokenizer = tokenizer,
    model = model,
    val_size = 0.15, # not used in combination with split csv
    epochs = 2,
    seed = 42
    )

In [ ]:
# SCORES
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import numpy as np

probabilities = fine_tuned_model.predict(fine_tuned_model.val_dataloader)
predictions = np.argmax(probabilities, axis=1)
true_labels = fine_tuned_model.val_dataloader.dataset.tensors[2].numpy()
f1 = f1_score(true_labels, predictions, average='macro')

print(f"The macro F1-score based on the validation set is: {f1:.4f} ({f1 * 100:.1f}%)")
print(classification_report(true_labels, predictions))
print(confusion_matrix(true_labels, predictions))

In [ ]:
# SAVING
if hasattr(fine_tuned_model.model, 'module'):
    model_to_save = fine_tuned_model.model.module
    print("module")
else:
    model_to_save = fine_tuned_model.model
    print("model")

output_dir = "./trained_bert10"

model_to_save.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
# LOADING
from transformers import BertForSequenceClassification, BertTokenizer, AutoModelForSequenceClassification, AutoTokenizer

saved_model = BertForSequenceClassification.from_pretrained("./trained_bert9")
saved_tokenizer = BertTokenizer.from_pretrained("./trained_bert9")

In [ ]:
# DETAILED SCORES FOR REPORT
import torch
from sklearn.metrics import classification_report
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "sentiment-analysis",
    model=saved_model,
    tokenizer=saved_tokenizer,
    device=device,
    batch_size=32,
)
df_test = df[(df["split"] == "test") & (df["review_cleaned"].notna())].copy()
y_true = df_test["sentiment_encoded"].tolist()

def review_generator():
    for review in df_test["review_cleaned"]:
        yield str(review)  # Forceer string-type voor de zekerheid

print(f"Generating {len(df_test)} predictions...")

y_pred = []
for res in classifier(review_generator(), truncation=True, max_length=512):
    y_pred.append(int(res["label"].split("_")[-1]))

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, digits=4))

df_test["y_pred"] = y_pred
df_tweets = df_test[df_test["type"] == "Tweet"]
print(f"Classification Report: TWEETS ({len(df_tweets)} items)")
if not df_tweets.empty:
    print(
        classification_report(
            df_tweets["sentiment_encoded"], df_tweets["y_pred"], digits=4
        )
    )
else:
    print("No Tweet data found.")

df_reviews = df_test[df_test["type"] == "Review"]
print(f"Classification Report: REVIEWS ({len(df_reviews)} items)")
if not df_reviews.empty:
    print(
        classification_report(
            df_reviews["sentiment_encoded"], df_reviews["y_pred"], digits=4
        )
    )
else:
    print("No Review data found.")

In [ ]:
# API PREPARATION
import pickle
import torch
import re
from bs4 import BeautifulSoup

class BertVectorizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def clean_text(self, x):
        x = BeautifulSoup(x, "html.parser").get_text()
        x = re.sub(r"http\S+|www\.\S+", "URL", x)
        x = re.sub(r"@\w+", "MNTN", x)
        x = emoji.demojize(x, delimiters=(" ", " ")).replace("_", " ")
        x = re.sub(r'\s+', ' ', x)
        return x.lower()

    def transform(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        
        cleaned_texts = [self.clean_text(t) for t in texts]
        
        batch_encoder = self.tokenizer(
            cleaned_texts,
            max_length=512,
            padding=True,  # "max_length"
            truncation=True,
            return_tensors='pt'
        )
        return batch_encoder

    def fit_transform(self, texts):
        return self.transform(texts)


class BertClassifier:
    def __init__(self, model):
        self.model = model
        self.device = torch.device(
            'cuda' if torch.cuda.is_available() else 'cpu'
        )

        self.model.to(self.device)
        self.model.eval()


    def predict(self, X_pred, batch_size=2):
        input_ids = X_pred['input_ids']
        attention_mask = X_pred['attention_mask']
        
        num_samples = input_ids.shape[0]
        
        predictions = []

        with torch.inference_mode():

            for i in range(0, num_samples, batch_size):

                ids = input_ids[i:i+batch_size].to(self.device)
                mask = attention_mask[i:i+batch_size].to(self.device)


                outputs = self.model(
                    input_ids=ids,
                    attention_mask=mask,
                    return_dict=False
                )

                logits = outputs[0]

                preds = torch.argmax(
                    logits,
                    dim=1
                ).cpu().tolist()

                predictions.extend(preds)

        return predictions


filename = 'benchmark.model'

deployment = {
    "vectorizer": BertVectorizer(saved_tokenizer),
    "classifier": BertClassifier(saved_model)
}

with open(filename, 'wb') as file:
    pickle.dump(deployment, file)

print(f"Model saved as '{filename}'!")